In [ ]:
from Crypto.Util.number import getPrime

p = getPrime(1024)
q = getPrime(1024)
if p > q:
	p, q = q, p
N = p * q

d = getPrime(500)
e = pow(d, -1, (p-1)*(q-1))

primes = [getPrime(256) for _ in range(9)]
e_residues = [e % p for p in primes]

key = sha256((str(p) + str(q)).encode()).digest()

So we know 

$e_i \equiv e \mod p_i$

$e \mod \prod_{i} p_i$

However, then we should be able to deduct that is $e \mod N$ as $gcd(N,\prod_{i} p_i) = 1$. If not one, we already solved it. However, after we know $d$, we need to get to $e$ such that we can get $p$ and $q$

In [4]:
N = 15645291629336676173452523665479966641549663029511406922591627966349033854046461193916717587062784418418884041906894266710723352555629413165733211432574714724004524581719819852210320366618663431805409883744870339931343799548264088641101056234665541662805740785444731719105742156418645804539912980832903937448988084182871628785371716750244543974032357033441587174836969377753263270293383555610133209593432713310685211342360724932526513920507234878706128843474723033064828822921829907489228281701101437766108791767842228037434455575223463208459227453856056734072769319409732570763152508416296704941384742389346373085251
primes = [94702418154069635448485990005812615732260709417810154777837589115235171329961, 74740302032029936941793075214256318553133624399259255906193982952217517713919, 102417040721788813587907777725516982121419224708689428235647237143694063731851, 110268372205108623872549220976509054352279361862090847192197359610994000662509, 114200462961848536195586358334261595124477794941339353745390412567741427558883, 76300261044316868422252787233676317217009570883746437967825127771465936890989, 91517240214721832452392104102512611974053170999965066779447192941005813525309, 77317657917806590177226699455009338787500605142123634032609118473192309822853, 64629840611708126358703960554236940650077675398378064917681587536063531344737]
e_residues = [73202545289090958105687519874164891582323813822403077825610814725240657081993, 47334322483582084750084183902844304846210665951321623916519242495793904908981, 76424012313967398382519263910578352948737432324795476941000596659630686417001, 1053045462242093797655189267336319584537875621561480110537943803387439305390, 31857341508561418414122204330395977484386970269736729202158964842913145088812, 30854214342268324841001078882562466126856145492568629680291251884285063089410, 71855994814286115581120401888066847776430565778117155851631781539231524733782, 65022384125458125147781746598237844903797802157282167031848529625742374276711, 45186685716073291045995972293483783519162221831221111538382446713678053488783]
ct = bytes.fromhex("e32fe4a8d6ac6aa94797cd338471c3c06311bf8836731779fa94208a862127aa2f714cb7e63e498eb1f81ab94161a72aa5bee26dcb8fbd90f3f3ca1dbb2c85ef")

In [1]:
from sage.all import *

In [8]:
P = reduce(lambda x,y:x*y,primes,1)
gcd(N,P)

1

In [15]:
e = CRT_list(
    e_residues,
    primes
)
ee = e % N 

assert ee == e

https://crypto.stanford.edu/~dabo/papers/RSA-survey.pdf:

- Try wiener, as e is huge
  
https://cryptohack.gitbook.io/cryptobook/untitled/low-private-component-attacks/wieners-attack:

In [20]:
from Crypto.Util.number import long_to_bytes

def wiener(e, n):
    # Convert e/n into a continued fraction
    cf = continued_fraction(e/n)
    convergents = cf.convergents()
    for kd in convergents:
        k = kd.numerator()
        d = kd.denominator()
        # Check if k and d meet the requirements
        if k == 0 or d%2 == 0 or e*d % k != 1:
            continue
        phi = (e*d - 1)/k
        # Create the polynomial
        x = PolynomialRing(RationalField(), 'x').gen()
        f = x**2 - (n-phi+1)*x + n
        roots = f.roots()
        # Check if polynomial as two roots
        if len(roots) != 2:
            continue
        # Check if roots of the polynomial are p and q
        p,q = int(roots[0][0]), int(roots[1][0])
        if p*q == n:
            return p,q
    return None

p,q = wiener(e,N)
assert p*q == N

In [22]:
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad
from hashlib import sha256

for a,b in [[p,q],[q,p]]:
    key = sha256((str(a) + str(b)).encode()).digest()
    print(AES.new(key, AES.MODE_ECB).decrypt(ct))

b'\xd4:\x04\x87\xe7\xd3.S\xe2\x06\xcf\x1c\x1c\x08\x88\xaa\xce\xd6\xc4\xd4\xbeDgr\xd9M6L\x9e\x01$\xf6f\x94T\x9a#&\x11!\x8b3\xc2\xf4\xcf\x1b\x8eFT\xa6\x8aS%\xa4\x92\r\x8f\xf3\xd3\xd9\x7fA&\xbd'
b"openECSC{0h_n0!M4yb3_I_sh0uldn'7_sw4p_e_4nd_d_4f73r_4ll...}\x05\x05\x05\x05\x05"
